1. Criar tabela queimadas_raw
2. Transformações nos dados -> criando outra tabela
3. Fazer uma análise de focos de incêndio

## criar tabela queimadas_raw

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.queimadas_raw
USING delta -- tabela final será Delta, habilita replace table
AS
SELECT * FROM read_files(
  '/Volumes/workspace/default/ingestion_data/queimadas.csv',
  format => 'csv',
  header => true,
  inferSchema => true,
  schemaEvolutionMode => 'none' -- desabilita _rescued_data col (schema inference)
);


In [0]:
%sql
SELECT * FROM workspace.default.queimadas_raw LIMIT 5;

## Lendo tabela -> datafrane

In [0]:
database_name = 'workspace.default'
table_name = 'queimadas_raw'

df_raw = spark.read.table(f'{database_name}.{table_name}')
display(df_raw.limit(100))

In [0]:
df_raw.show(100)

## Limpeza de dados

ajustes de null

In [0]:
from pyspark.sql.functions import col

df_raw.filter(col("dias_sem_chuva") == -999).count()

In [0]:
# quantidade de linhas sem dados de dias de chuva por localidade (long, lat)

display(
    df_raw.filter(col("dias_sem_chuva") == -999)
    # .groupBy("latitude", "longitude", "sigla_uf")
    .groupBy("sigla_uf")
    .count()
    .orderBy(col("count"), ascending=False)
)

In [0]:
# converter pra null

df_cleaned = df_raw.replace(-999, None, "dias_sem_chuva")

df_cleaned.filter(col("dias_sem_chuva").isNull()).count()


In [0]:
display(df_cleaned.limit(100))

criando PK

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

df_pk = df_cleaned.withColumn("id", monotonically_increasing_id())

# row_number().over(Window.orderBy("data_hora", "latitude", "longitude"))

display(df_pk.limit(100))

## Escrevendo tabela

In [0]:
path_table = 'workspace.default'
table_name = 'queimadas_cleaned'

df_pk.write.mode("overwrite").saveAsTable(f'{path_table}.{table_name}')

In [0]:
df_result = spark.read.table(f'{path_table}.{table_name}')

display(df_result.limit(100))

# O que evitar 

In [0]:
.toPandas() -> pandas.DataFrame -> adeus, paralelismo! 
.collect() -> lista -> adeus, paralelismo! OOM ou OutOfMemoryError